In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from scipy.stats import mode
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

In [ ]:
url = "../data/features/test_sr_44100.csv"
column = ['id', 'spectral_centroid', 'spectral_skewness', 'spectral_kurtosis',\
    'spectral_entropy', 'spectral_spread', 'spectral_flatness', 'spectral_rolloff', 'spectral_flux',\
    'spectral_mean', 'spectral_rms', 'spectral_std', 'spectral_variance', 'spectrum', \
    'mean_frequency', 'peak_frequency', 'frequencies_std', 'amplitudes_cum_sum', 'mode_frequency', \
    'median_frequency', 'frequencies_q25', 'frequencies_q75', 'iqr', 'freqs_skewness', \
    'freqs_kurtosis', 'energy', 'rms', 'zcr', 'meanfun', \
    'minfun', 'maxfun', 'meandom', 'mindom', 'maxdom', \
    'dfrange', 'modindex', 'signal', 'mfcc', 'bfcc', \
    'lfcc', 'lpc', 'lpcc', 'msrcc', 'ngcc', 'psrcc', \
    'plp', 'rplp', 'gfcc']

df = pd.read_csv(url, names = column)
df = df.iloc[1:]
df = df.set_index('id')

In [ ]:
def process_complex(x):
    try:
        if isinstance(x, str) and '(' in x and ')' in x:
            # 문자열에서 괄호를 제거하고 복소수로 변환한 후 실수 부분 추출
            x = complex(x.strip('()'))
            return x.real
        elif isinstance(x, complex):
            return x.real
        else:
            return float(x)
    except ValueError:
        return np.nan

for column in df.columns:
    df[column] = df[column].apply(process_complex)

In [ ]:
for col in df.columns[1:-1]:  # 첫 번째 열 제외, 마지막 열 제외
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1

    min_threshold = df[col].min()- 1.5 * IQR
    max_threshold = df[col].max() + 1.5 * IQR

    # Calculate the mode Series
    mode_result = mode(df[col])
    # Assuming there's only one mode (most frequent value)
    most_frequent = mode_result.mode
    
    # IQR 범위를 벗어나는 값들을 최빈값으로 대체
    df[col] = np.where(df[col] < min_threshold, most_frequent, df[col])
    df[col] = np.where(df[col] > max_threshold, most_frequent, df[col])

print(df)

In [ ]:
df.info()

In [ ]:


# 데이터 타입 확인 및 변환
for column in df.columns[1:-1]:
    if df[column].dtype != 'float':
        df[column] = df[column].astype(float)

# 서브플롯의 크기 
num_columns = 46
n_cols = 4  # 한 행에 표시할 열 수
n_rows = (num_columns + n_cols - 1) // n_cols  # 총 행 수 계산
fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, n_rows * 3))

# # 2번째 열부터 마지막 열까지 분포 그리기
# for i, column in enumerate(df.columns[1:], start=1):  # 첫 번째 열 제외, 마지막 열 제외
    
#     sns.histplot(hue='label', multiple='stack', palette='pastel', ax=ax)
#     ax.set_title(column)
#     ax.set_xlabel('')
#     ax.set_ylabel('')
#     ax.tick_params(axis='x', rotation=45)
#     ax.xaxis.set_major_locator(plt.MaxNLocator(10))  # x축 틱을 최대 10개로 설정



for i, col in enumerate(df.columns, start = 1):
    ax = axes[(i - 1) // n_cols, (i - 1) % n_cols]  # 서브플롯의 위치 계산
    sns.histplot(data=df, x=col, kde=True, bins=100, ax=ax)
    ax.set_title(col)
    ax.set_xlabel('')
    ax.set_ylabel('')
    ax.tick_params(axis='x', rotation=45)
    ax.xaxis.set_major_locator(plt.MaxNLocator(10))  # x축 틱을 최대 10개로 설정

plt.tight_layout()
plt.show()

# 마지막 빈 서브플롯 제거
for j in range(i, n_rows * n_cols):
    fig.delaxes(axes[j // n_cols, j % n_cols])

plt.tight_layout()
plt.show()

In [ ]:
df = df.drop(['freqs_skewness','dfrange', 'modindex', 'mean_frequency', 'frequencies_std', 'median_frequency', \
    'freqs_kurtosis'], axis=1) # 0인 값,  drop

In [ ]:
x = df.values

In [ ]:
feature_name_X = ['spectral_centroid', 'spectral_skewness', \
    'spectral_kurtosis', 'spectral_entropy', 'spectral_spread', 'spectral_flatness', \
    'spectral_rolloff', 'spectral_flux', 'spectral_mean', 'spectral_rms',\
    'spectral_std', 'spectral_variance', 'spectrum', 'peak_frequency', \
    'amplitudes_cum_sum', 'mode_frequency',\
    'frequencies_q25', 'frequencies_q75', 'iqr',\
    'energy', 'rms', 'zcr', 'meanfun', 'minfun', 'maxfun', 'meandom', 'mindom', 'maxdom', \
    'signal', 'mfcc', 'bfcc', 'lfcc', 'lpc', 'lpcc', 'msrcc', \
    'ngcc', 'psrcc', 'plp', 'rplp', 'gfcc']

In [ ]:
# 독립 변인 표준화 및 데이터 저장 
x_scaled = StandardScaler().fit_transform(x)

X = pd.DataFrame(x_scaled, columns = feature_name_X)
X.index = df.index

#PCA
pca = PCA(n_components=20)
printcipalComponents = pca.fit_transform(x)
printcipalDf = pd.DataFrame(data = printcipalComponents)
printcipalDf.columns = pca.get_feature_names_out()
printcipalDf.index = X.index
# PCA 설명
print("Explained variance ratio:")
print(pca.explained_variance_ratio_)
print("Total variance explained by the first two components:")
print(sum(pca.explained_variance_ratio_))

In [ ]:
printcipalDf.to_csv('PCA_44100_train.csv', index = False) 